# Strong Evaluator Results

This notebook summarizes evaluator-backend robustness results after adding larger or stronger judge backends.

本 notebook 总结加入更大或更强 evaluator backend 之后的 robustness results。

The target responses remain fixed; only the evaluator backend changes.

target responses 保持不变，变化的只是 evaluator backend。


## Experiment Question

The main experiment shows that CALE behavior has interpretable internal structure under the heuristic evaluator.

主实验显示，在 heuristic evaluator 下，CALE behavior 具有可解释的内部结构。

This extension asks whether the structured-evaluation advantage remains visible when the evaluator backend changes to Qwen2.5-7B or DeepSeek V4-Pro.

这个 extension 问的是：当 evaluator backend 换成 Qwen2.5-7B 或 DeepSeek V4-Pro 时，structured evaluation 的优势是否仍然可见。


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from analyze_behavior_matrix import numeric_behavior_columns, run_pca
from visualize_behavior_matrix import CORE_BEHAVIOR_COLUMNS, PROXY_COLUMNS, existing_numeric_columns, save_heatmap

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 220)

OUTDIR = Path('figures/strong_evaluator_results')
OUTDIR.mkdir(parents=True, exist_ok=True)

RESULTS = {
    'heuristic_full': {
        'label': 'Heuristic full evaluator',
        'path': Path('outputs/small_models_all/fever_dev_qwen25_15b_llama32_1b_neutral_full_eval_behavior_matrix.csv'),
        'expected_rows': 239976,
        'kind': 'full_main',
    },
    'qwen25_7b_limit1000': {
        'label': 'Qwen2.5-7B local evaluator',
        'path': Path('outputs/small_models_all/fever_dev_qwen25_15b_llama32_1b_neutral_strong_qwen25_7b_limit1000_eval_behavior_matrix.csv'),
        'expected_rows': 2000,
        'kind': 'strong_subset',
    },
    'deepseek_v4_pro_limit1000': {
        'label': 'DeepSeek V4-Pro API evaluator',
        'path': Path('outputs/small_models_all/fever_dev_qwen25_15b_llama32_1b_neutral_deepseek_v4_pro_limit1000_eval_behavior_matrix.csv'),
        'expected_rows': 2000,
        'kind': 'strong_subset',
    },
}

def pretty(name):
    return str(name).replace('_proxy', '').replace('_', ' ').title()

def load_matrix(key, spec):
    path = spec['path']
    if not path.exists():
        return None
    df = pd.read_csv(path)
    df['evaluator_backend'] = key
    df['evaluator_backend_label'] = spec['label']
    return df

def existing_behavior_columns(df):
    return existing_numeric_columns(df, CORE_BEHAVIOR_COLUMNS) + existing_numeric_columns(df, PROXY_COLUMNS)

def save_bar(table, path, title, ylabel='value'):
    fig, ax = plt.subplots(figsize=(max(7, 0.7 * len(table.index)), 4.8))
    table.plot(kind='bar', ax=ax)
    ax.set_title(title)
    ax.set_ylabel(ylabel)
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=25)
    ax.legend(loc='best', fontsize=8)
    fig.tight_layout()
    fig.savefig(path, dpi=240)
    plt.show()


## 1. File And Run Audit

We first check which result files are available and whether row counts match the intended experiment design.

我们首先检查哪些结果文件已经存在，以及行数是否符合实验设计。


In [ ]:
audit_rows = []
matrices = {}
for key, spec in RESULTS.items():
    df = load_matrix(key, spec)
    if df is not None:
        matrices[key] = df
    audit_rows.append({
        'backend': key,
        'label': spec['label'],
        'path': str(spec['path']),
        'exists': spec['path'].exists(),
        'rows': len(df) if df is not None else 0,
        'expected_rows': spec['expected_rows'],
        'row_count_ok': (len(df) == spec['expected_rows']) if df is not None else False,
        'variants': ', '.join(sorted(df['variant'].dropna().astype(str).unique())) if df is not None and 'variant' in df else '',
        'target_models': ', '.join(sorted(df['model_name'].dropna().astype(str).unique())) if df is not None and 'model_name' in df else '',
    })
audit = pd.DataFrame(audit_rows)
audit.to_csv(OUTDIR / 'file_audit.csv', index=False)
audit


## 2. Direct Judge Versus Full CALE Outcomes

This comparison uses shared outcome variables available to both direct judge and full CALE.

这个比较只使用 direct judge 和 full CALE 都有的 shared outcome variables。


In [ ]:
outcome_rows = []
for key, df in matrices.items():
    if 'variant' not in df.columns:
        continue
    grouped = df.groupby('variant', dropna=False)
    for variant, group in grouped:
        row = {
            'backend': key,
            'backend_label': RESULTS[key]['label'],
            'variant': variant,
            'rows': len(group),
        }
        if 'final_score' in group:
            row['mean_final_score'] = pd.to_numeric(group['final_score'], errors='coerce').mean()
        elif 'score' in group:
            row['mean_final_score'] = pd.to_numeric(group['score'], errors='coerce').mean()
        if 'uncertainty' in group:
            row['mean_uncertainty'] = pd.to_numeric(group['uncertainty'], errors='coerce').mean()
        outcome_rows.append(row)
outcome_summary = pd.DataFrame(outcome_rows)
outcome_summary.to_csv(OUTDIR / 'direct_vs_cale_outcome_summary.csv', index=False)
outcome_summary


In [ ]:
if not outcome_summary.empty and 'mean_final_score' in outcome_summary:
    plot_df = outcome_summary.pivot(index='backend_label', columns='variant', values='mean_final_score')
    plot_df.to_csv(OUTDIR / 'direct_vs_cale_mean_score_pivot.csv')
    save_bar(plot_df, OUTDIR / 'direct_vs_cale_mean_score.png', 'Mean Final Score by Evaluator Backend and Variant', ylabel='mean final score')
    plot_df.round(3)


## 3. Full CALE Behavior Profile By Backend

Here we compare only `full_attack_aware_cale`, because direct judge does not produce construct-level CALE behavior variables.

这里我们只比较 `full_attack_aware_cale`，因为 direct judge 不产生 construct-level CALE behavior variables。


In [ ]:
profile_rows = []
for key, df in matrices.items():
    if 'variant' not in df.columns:
        continue
    full = df[df['variant'].eq('full_attack_aware_cale')].copy()
    if full.empty:
        continue
    cols = existing_behavior_columns(full)
    row = {'backend': key, 'backend_label': RESULTS[key]['label'], 'rows': len(full)}
    for col in cols:
        row[col] = pd.to_numeric(full[col], errors='coerce').mean()
    profile_rows.append(row)
backend_profile = pd.DataFrame(profile_rows).set_index('backend_label') if profile_rows else pd.DataFrame()
backend_profile.to_csv(OUTDIR / 'full_cale_behavior_profile_by_backend.csv')
backend_profile.round(3)


In [ ]:
if not backend_profile.empty:
    heat_cols = [c for c in CORE_BEHAVIOR_COLUMNS + PROXY_COLUMNS if c in backend_profile.columns]
    save_heatmap(
        backend_profile[heat_cols],
        OUTDIR / 'full_cale_behavior_profile_by_backend.png',
        'Full CALE Behavior Profile by Evaluator Backend',
        vmin=0,
        vmax=1,
        cmap='viridis',
    )
    backend_profile[heat_cols].round(3)


## 4. PCA Structure By Backend

PCA is run only on `full_attack_aware_cale` rows for each backend, because direct judge rows have missing construct variables.

PCA 只在每个 backend 的 `full_attack_aware_cale` rows 上运行，因为 direct judge rows 缺少 construct variables。

Interpret PCA signs cautiously; signs are arbitrary, while loading patterns are meaningful.

解释 PCA 时要小心符号；符号本身是任意的，loading pattern 才有意义。


In [ ]:
pca_summary_rows = []
top_loading_rows = []
for key, df in matrices.items():
    if 'variant' in df.columns:
        pca_df = df[df['variant'].eq('full_attack_aware_cale')].copy()
    else:
        pca_df = df.copy()
    if len(pca_df) < 5:
        continue
    columns = numeric_behavior_columns(pca_df, include_final_score=False)
    selected = pca_df[columns].apply(pd.to_numeric, errors='coerce')
    try:
        loadings, variance, scores = run_pca(selected, n_components=4, max_missing_share=0.5)
    except ValueError as exc:
        pca_summary_rows.append({'backend': key, 'backend_label': RESULTS[key]['label'], 'error': str(exc)})
        continue
    variance.to_csv(OUTDIR / f'{key}_pca_explained_variance.csv', index=False)
    loadings.to_csv(OUTDIR / f'{key}_pca_loadings.csv')
    pca_summary_rows.append({
        'backend': key,
        'backend_label': RESULTS[key]['label'],
        'rows': len(pca_df),
        'pc1_variance': variance.loc[variance['component'].eq('PC1'), 'explained_variance_ratio'].iloc[0] if 'PC1' in set(variance['component']) else np.nan,
        'pc1_pc4_cumulative_variance': variance['explained_variance_ratio'].sum(),
        'pc1_top_variables': ', '.join(loadings['PC1'].abs().sort_values(ascending=False).head(4).index.tolist()) if 'PC1' in loadings else '',
        'pc2_top_variables': ', '.join(loadings['PC2'].abs().sort_values(ascending=False).head(4).index.tolist()) if 'PC2' in loadings else '',
    })
    for component in loadings.columns:
        ordered = loadings[component].sort_values(key=lambda s: s.abs(), ascending=False)
        for rank, (variable, loading) in enumerate(ordered.head(8).items(), start=1):
            top_loading_rows.append({
                'backend': key,
                'backend_label': RESULTS[key]['label'],
                'component': component,
                'rank': rank,
                'variable': variable,
                'loading': loading,
                'absolute_loading': abs(loading),
            })

pca_summary = pd.DataFrame(pca_summary_rows)
pca_top_loadings = pd.DataFrame(top_loading_rows)
pca_summary.to_csv(OUTDIR / 'backend_pca_summary.csv', index=False)
pca_top_loadings.to_csv(OUTDIR / 'backend_pca_top_loadings.csv', index=False)
pca_summary


In [ ]:
if not pca_summary.empty and 'pc1_pc4_cumulative_variance' in pca_summary:
    var_plot = pca_summary.set_index('backend_label')[['pc1_variance', 'pc1_pc4_cumulative_variance']]
    save_bar(var_plot, OUTDIR / 'backend_pca_variance_summary.png', 'PCA Variance Summary by Evaluator Backend', ylabel='explained variance')
    var_plot.round(3)


## 5. Paper-Facing Backend Comparison Table

This table is the main artifact for writing: it compares backend role, sample size, PC1 interpretation, and the direct-vs-CALE finding.

这张表是写作时最主要的 artifact：它比较 backend role、sample size、PC1 interpretation 和 direct-vs-CALE finding。


In [ ]:
paper_rows = []
for _, row in audit.iterrows():
    backend = row['backend']
    pca_row = pca_summary[pca_summary['backend'].eq(backend)].iloc[0].to_dict() if not pca_summary.empty and pca_summary['backend'].eq(backend).any() else {}
    outcome_backend = outcome_summary[outcome_summary['backend'].eq(backend)] if not outcome_summary.empty else pd.DataFrame()
    direct_score = np.nan
    cale_score = np.nan
    if not outcome_backend.empty and 'mean_final_score' in outcome_backend:
        direct_vals = outcome_backend[outcome_backend['variant'].eq('direct_llm_judge')]['mean_final_score']
        cale_vals = outcome_backend[outcome_backend['variant'].eq('full_attack_aware_cale')]['mean_final_score']
        if not direct_vals.empty:
            direct_score = direct_vals.iloc[0]
        if not cale_vals.empty:
            cale_score = cale_vals.iloc[0]
    paper_rows.append({
        'backend': backend,
        'backend_label': row['label'],
        'rows': row['rows'],
        'variants': row['variants'],
        'row_count_ok': row['row_count_ok'],
        'direct_mean_score': direct_score,
        'full_cale_mean_score': cale_score,
        'pc1_pc4_cumulative_variance': pca_row.get('pc1_pc4_cumulative_variance', np.nan),
        'pc1_top_variables': pca_row.get('pc1_top_variables', ''),
        'interpretation_note': 'TBD after inspecting backend-specific loadings' if row['exists'] else 'waiting for result file',
    })
paper_table = pd.DataFrame(paper_rows)
paper_table.to_csv(OUTDIR / 'paper_backend_comparison_table.csv', index=False)
paper_table


## 6. Combined Evaluator-Backend View

This section combines available evaluator backends into one table so we can compare heuristic, larger local, and API evaluators side by side.

这一节把已有 evaluator backends 合在一张表里，方便并排比较 heuristic、larger local 和 API evaluators。

Because backend scales and sample sizes differ, interpret this as descriptive alignment, not as a calibrated leaderboard.

因为 backend 的尺度和样本量不同，这里应该解释为 descriptive alignment，而不是 calibrated leaderboard。


In [ ]:
if matrices:
    combined = pd.concat(matrices.values(), ignore_index=True, sort=False)
else:
    combined = pd.DataFrame()
combined.to_csv(OUTDIR / 'combined_available_behavior_matrix.csv', index=False)

combined_summary_rows = []
if not combined.empty:
    score_col = 'final_score' if 'final_score' in combined.columns else 'score' if 'score' in combined.columns else None
    group_cols = ['evaluator_backend_label', 'variant']
    for keys, group in combined.groupby(group_cols, dropna=False):
        backend_label, variant = keys
        row = {
            'backend_label': backend_label,
            'variant': variant,
            'rows': len(group),
            'target_models': ', '.join(sorted(group['model_name'].dropna().astype(str).unique())) if 'model_name' in group else '',
            'domains': ', '.join(sorted(group['domain'].dropna().astype(str).unique())) if 'domain' in group else '',
        }
        if score_col:
            row['mean_score'] = pd.to_numeric(group[score_col], errors='coerce').mean()
        if 'uncertainty' in group:
            row['mean_uncertainty'] = pd.to_numeric(group['uncertainty'], errors='coerce').mean()
        combined_summary_rows.append(row)
combined_backend_variant_summary = pd.DataFrame(combined_summary_rows)
combined_backend_variant_summary.to_csv(OUTDIR / 'combined_backend_variant_summary.csv', index=False)
combined_backend_variant_summary


In [ ]:
if not combined_backend_variant_summary.empty and 'mean_score' in combined_backend_variant_summary:
    combined_score_pivot = combined_backend_variant_summary.pivot(index='backend_label', columns='variant', values='mean_score')
    combined_score_pivot.to_csv(OUTDIR / 'combined_backend_variant_mean_score_pivot.csv')
    save_bar(combined_score_pivot, OUTDIR / 'combined_backend_variant_mean_score.png', 'Mean Score Across Available Evaluator Backends', ylabel='mean score')
    combined_score_pivot.round(3)


## 7. Target-Model Dominance Check

This section asks whether the observed evaluator-backend structure is dominated by one response-generating target model.

这一节问的是：当前 evaluator-backend structure 会不会被某一个生成回答的 target model 主导。

For each evaluator backend, we compare pooled, Qwen-target-only, and Llama-target-only splits.

对于每个 evaluator backend，我们比较 pooled、Qwen-target-only 和 Llama-target-only 三种 split。

Domain counts are still reported as a secondary audit, but the current FEVER neutral run is expected to be mostly `general` domain.

domain counts 仍然作为次要 audit 输出，但当前 FEVER neutral run 预期基本都是 `general` domain。


In [ ]:
slice_rows = []
if not combined.empty:
    score_col = 'final_score' if 'final_score' in combined.columns else 'score' if 'score' in combined.columns else None
    for slice_col in ['model_name', 'domain', 'dataset']:
        if slice_col not in combined.columns:
            continue
        for keys, group in combined.groupby(['evaluator_backend_label', 'variant', slice_col], dropna=False):
            backend_label, variant, slice_value = keys
            row = {
                'slice': slice_col,
                'backend_label': backend_label,
                'variant': variant,
                'slice_value': slice_value,
                'rows': len(group),
            }
            if score_col:
                row['mean_score'] = pd.to_numeric(group[score_col], errors='coerce').mean()
            for col in CORE_BEHAVIOR_COLUMNS:
                if col in group.columns:
                    row[col] = pd.to_numeric(group[col], errors='coerce').mean()
            slice_rows.append(row)
slice_summary = pd.DataFrame(slice_rows)
slice_summary.to_csv(OUTDIR / 'target_model_domain_slice_summary.csv', index=False)
slice_summary.head(30)


In [ ]:
domain_counts = pd.DataFrame()
if not combined.empty and 'domain' in combined.columns:
    domain_counts = combined.groupby(['evaluator_backend_label', 'domain'], dropna=False).size().unstack(fill_value=0)
    domain_counts.to_csv(OUTDIR / 'domain_counts_by_backend.csv')

target_counts = pd.DataFrame()
if not combined.empty and 'model_name' in combined.columns:
    target_counts = combined.groupby(['evaluator_backend_label', 'model_name'], dropna=False).size().unstack(fill_value=0)
    target_counts.to_csv(OUTDIR / 'target_model_counts_by_backend.csv')

print('Domain counts by backend:')
display(domain_counts)
print('Target-model counts by backend:')
display(target_counts)


### PCA Dominance Check By Target Model

We rerun PCA within each target-model split using only `full_attack_aware_cale` rows.

我们只使用 `full_attack_aware_cale` rows，在每个 target-model split 内重新运行 PCA。

If PC1 top variables remain similar across Qwen-target and Llama-target splits, the behavior structure is less likely to be dominated by one response model.

如果 Qwen-target 和 Llama-target splits 的 PC1 top variables 保持相似，那么 behavior structure 就不太可能被某一个回答模型主导。


In [ ]:
dominance_rows = []
dominance_loading_rows = []

def target_split_name(model_name):
    value = str(model_name).lower()
    if 'qwen' in value:
        return 'target_qwen_only'
    if 'llama' in value:
        return 'target_llama_only'
    return 'target_other'

for key, df in matrices.items():
    if 'variant' not in df.columns or 'model_name' not in df.columns:
        continue
    full = df[df['variant'].eq('full_attack_aware_cale')].copy()
    if full.empty:
        continue
    full['target_split'] = full['model_name'].map(target_split_name)
    split_items = [('pooled', full)] + list(full.groupby('target_split', dropna=False))
    for split_name, split_df in split_items:
        if len(split_df) < 5:
            dominance_rows.append({
                'backend': key,
                'backend_label': RESULTS[key]['label'],
                'target_split': split_name,
                'rows': len(split_df),
                'error': 'too few rows for PCA',
            })
            continue
        columns = numeric_behavior_columns(split_df, include_final_score=False)
        selected = split_df[columns].apply(pd.to_numeric, errors='coerce')
        try:
            loadings, variance, _ = run_pca(selected, n_components=4, max_missing_share=0.5)
        except ValueError as exc:
            dominance_rows.append({
                'backend': key,
                'backend_label': RESULTS[key]['label'],
                'target_split': split_name,
                'rows': len(split_df),
                'error': str(exc),
            })
            continue
        dominance_rows.append({
            'backend': key,
            'backend_label': RESULTS[key]['label'],
            'target_split': split_name,
            'rows': len(split_df),
            'pc1_variance': variance.loc[variance['component'].eq('PC1'), 'explained_variance_ratio'].iloc[0] if 'PC1' in set(variance['component']) else np.nan,
            'pc1_pc4_cumulative_variance': variance['explained_variance_ratio'].sum(),
            'pc1_top_variables': ', '.join(loadings['PC1'].abs().sort_values(ascending=False).head(4).index.tolist()) if 'PC1' in loadings else '',
            'pc2_top_variables': ', '.join(loadings['PC2'].abs().sort_values(ascending=False).head(4).index.tolist()) if 'PC2' in loadings else '',
        })
        for component in loadings.columns:
            ordered = loadings[component].sort_values(key=lambda s: s.abs(), ascending=False)
            for rank, (variable, loading) in enumerate(ordered.head(8).items(), start=1):
                dominance_loading_rows.append({
                    'backend': key,
                    'backend_label': RESULTS[key]['label'],
                    'target_split': split_name,
                    'component': component,
                    'rank': rank,
                    'variable': variable,
                    'loading': loading,
                    'absolute_loading': abs(loading),
                })

target_model_dominance_pca = pd.DataFrame(dominance_rows)
target_model_dominance_loadings = pd.DataFrame(dominance_loading_rows)
target_model_dominance_pca.to_csv(OUTDIR / 'target_model_dominance_pca_summary.csv', index=False)
target_model_dominance_loadings.to_csv(OUTDIR / 'target_model_dominance_pca_top_loadings.csv', index=False)
target_model_dominance_pca


In [ ]:
if not target_model_dominance_pca.empty and 'pc1_pc4_cumulative_variance' in target_model_dominance_pca:
    dominance_var = target_model_dominance_pca.pivot(index='backend_label', columns='target_split', values='pc1_pc4_cumulative_variance')
    dominance_var.to_csv(OUTDIR / 'target_model_dominance_variance_pivot.csv')
    save_bar(dominance_var, OUTDIR / 'target_model_dominance_variance.png', 'PC1-PC4 Variance by Backend and Target-Model Split', ylabel='PC1-PC4 cumulative variance')
    dominance_var.round(3)


## 8. Takeaways Template

Use this section after both strong evaluator jobs finish.

等两个 strong evaluator jobs 都跑完后，再使用这一节。

Expected interpretation if the results align: CALE's diagnostic advantage is not specific to the heuristic evaluator, because full CALE continues to expose construct-level behavior profiles under larger local and API evaluator backends.

如果结果一致，预期解释是：CALE 的 diagnostic advantage 不只是 heuristic evaluator 的产物，因为在 larger local 和 API evaluator backends 下，full CALE 仍然能暴露 construct-level behavior profiles。


In [ ]:
summary_lines = []
for _, row in paper_table.iterrows():
    summary_lines.append(
        f"- {row['backend_label']}: rows={row['rows']}, variants={row['variants']}, "
        f"PC1-PC4 variance={row['pc1_pc4_cumulative_variance'] if pd.notna(row['pc1_pc4_cumulative_variance']) else 'NA'}, "
        f"PC1 top variables={row['pc1_top_variables'] or 'NA'}."
    )
summary_text = chr(10).join(summary_lines)
(OUTDIR / 'strong_evaluator_takeaways.md').write_text(summary_text, encoding='utf-8')
print(summary_text)
